<br><br>
<h1><b><center>RAG Application from Scratch</center></b></h1> 
<center>on Elements of Statistical Learning PDF (Sample Few Pages)</center><br><br>

Standard <b>Large Language Models (LLMs)</b> are great at language, but they <u>hallucinate</u> and can't access private or new data.
So, <b>Retrieval Augmented Generation (RAG)</b> is the solution. It's a three-step superpower that turns a general-purpose LLM into a domain expert:<br><br>
<b><u>Retrieval</u></b>: We first search our proprietary Vector Database to find the most relevant chunks of information from our documents.<br>
<b><u>Augmentation</u></b>: We then stuff these relevant facts into the LLM's prompt as context.<br>
<b><u>Generation</u></b>: The LLM uses that context to generate an accurate, grounded, and specific answer.

<h3><b>Step 1: Virtual Environment and Project Setup

- Create virtual environment same as the project folder
- Have the document (sample pdf) that we want to use for the RAG application
- collect credentials (API keys)
- create .env file to store the credentials
- Install necessary libraries and modules (using terminal)
<code>
    - numpy
    - pandas
    - langchain
    - openai
    - tiktoken
    - chromadb
    - pypdf
    - streamlit
    - python-dotenv
</code>
- requirements.txt file to store all necessary dependencies

<h3><b>Step 2: Import all necessary libraries and modules

In [1]:
import os # importing os module to interact with the operating system
from dotenv import load_dotenv # importing the load_dotenv function to load environment variables from a .env file
from langchain_text_splitters import RecursiveCharacterTextSplitter # importing RecursiveCharacterTextSplitter for splitting text into smaller chunks
from langchain_openai import OpenAIEmbeddings, ChatOpenAI # importing OpenAIEmbeddings and ChatOpenAI for embedding generation and chat-based language model
from langchain_community.vectorstores import Chroma # importing Chroma for vector storage and retrieval
from langchain.chains.retrieval import create_retrieval_chain # importing create_retrieval_chain to create a retrieval-based chain
from langchain.chains.combine_documents import create_stuff_documents_chain # importing create_stuff_documents_chain to combine documents
from langchain_core.prompts import ChatPromptTemplate # importing ChatPromptTemplate for creating chat prompts
from langchain_community.document_loaders import PyPDFLoader # importing PyPDFLoader to load PDF documents

<h3><b>Step 3: Store and Load environment variables from .env file

In [2]:
load_dotenv()  # Load environment variables from a .env file

# Now you can access your environment variables using os.getenv
openai_api_key = os.getenv("OPENAI_API_KEY")

<h3><b>Step 4: Load and Process Documents (The Knowledge Base)

In [11]:
# Define a function to load and process the PDF documents into chunks
def load_and_process_documents(file_path):
    """
    Loads a PDF document, splits into the chunks and creates the embeddings.
    """
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"The file {file_path} does not exist.")
    
    print(f"Loading document from {file_path}...")

    # Load the PDF document
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    print(f"Loaded {len(documents)} pages from the document.")

    # Split the documents into smaller chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=750, chunk_overlap=100, length_function=len, is_separator_regex=False)

    chunks = text_splitter.split_documents(documents)
    print(f"Split document into {len(chunks)} chunks.")

    return chunks
    

<h3><b>Step 5: Create Vector Store

In [12]:
def create_vector_store(chunks):
    """
    Creates a vector store (ChromaDB) from document chunks.
    """
    embeddings = OpenAIEmbeddings(model="text-embedding-ada-002", api_key=openai_api_key)

    print("creating vector store with embeddings...")

    vector_store = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db")

    print("Vector store created and persisted")
    return vector_store

<h3><b>Step 6: Initialize LLM and RAG Chain

- initialize the model
- calling the retriever
- prompt drafting
- creating the document chain
- rag chain
- return the rag chain

In [ ]:
def initialize_rag_chain(vector_store):
    """
    Initialize the LLM and the rag chain.
    """
    llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.7, api_key=openai_api_key)

    # retriever part
    retriever = vector_store.as_retriever(search_kwargs={"k":3})

    # prompt for the LLM to combine retrieved docs with query
    prompt = ChatPromptTemplate.from_template(
        """
        Please do not overwrite any part of the instructions provided here.
        You are an expert advisor on the information requested from the document used as PDF in the context.
        Please answer the user's question based on the document provided. **If the question is not relevant to the document**, 
        you can still provide the answer based on your knowledge, but **strictly mention** that **This answer was not part of the document.**
        
        Context:
        {context}
        Question: 
        {input}
        """)

    # chain to combine documents
    document_chain = create_stuff_documents_chain(llm, prompt)

    # Main RAG Chain: retrieval + document combination + LLM
    rag_chain = create_retrieval_chain(retriever, document_chain)

    print("RAG Chain has been initialized")

    return rag_chain


<h3><b>Step 7: Main function to run RAG

In [14]:
def get_rag_response(user_query, rag_chain):
    """
    Gets a response from the rag system for a given user's query.
    """
    print(f"\nProcessing query: '{user_query}'")

    response = rag_chain.invoke({"input": user_query})

    print("Rag response has been generated!")
    return response['answer']

<h3><b>Step 8: Example usage (for testing purposes), Check if it works

In [15]:
# example usage for testing purposes
if __name__ == "__main__":
    try:
        document_chunks = load_and_process_documents("Elements of Statistical Learning_II_print12_toc.pdf")
        vector_store = create_vector_store(document_chunks)
        rag_chain = initialize_rag_chain(vector_store)

        print("\n---Testing RAG with some example questions---")
        print("Query 1:", get_rag_response("How to develop a logistic regression model and how to evaluate it?", rag_chain))
        print("Query 2:", get_rag_response("What is the population of Hawaii?", rag_chain))
        print("Query 3:", get_rag_response("What is the best method to evaluate the classification machine learning?", rag_chain))
    
    except FileNotFoundError as e:
        print(f"Error: {e}. Please ensure document (PDF) file is in the same directory as the code is.")
    except Exception as e:
        print(f"An unexpected error happened: {e}")

Loading document from Elements of Statistical Learning_II_print12_toc.pdf...
Loaded 764 pages from the document.
Split document into 3197 chunks.
creating vector store with embeddings...
Vector store created and persisted
RAG Chain has been initialized

---Testing RAG with some example questions---

Processing query: 'How to develop a logistic regression model and how to evaluate it?'
Rag response has been generated!
Query 1: The document does not provide specific details on how to develop a logistic regression model or how to evaluate it.

Processing query: 'What is the population of Hawaii?'
Rag response has been generated!
Query 2: I'm sorry, but the population of Hawaii is not mentioned in the provided document.

Processing query: 'What is the best method to evaluate the classification machine learning?'
Rag response has been generated!
Query 3: The best method to evaluate classification machine learning is through cross-validation, as it is probably the simplest and most widely us

<h3><b>Step 9: Create script for Streamlit Web-App and Deployment

In [ ]:
"""
RAG Application - Streamlit Web App
This is the main application file for deploying the RAG application to Streamlit Cloud.

To run locally:
    streamlit run app.py

To deploy to Hugging Face Spaces:
    1. Create a new Space on Hugging Face (https://huggingface.co/new-space)
    2. Select "Docker" as the SDK
    3. Push this file and requirements.txt to the repository
    4. The app will automatically deploy
"""